In [2]:
    
import tkinter as tk
from tkinter import ttk, messagebox
from ddgs import DDGS
import threading
import webbrowser
import re
from urllib.parse import urlparse


# ============================================================
# GET PRICE FROM TEXT
# ============================================================

def extract_price(text):

    if not text:
        return None

    # Example: ₹39,999
    match = re.search(
        r'₹\s*([\d,]+)',
        text
    )

    if match:
        return int(
            match.group(1).replace(",", "")
        )

    # Example: Rs 39999 / Rs. 39,999
    match = re.search(
        r'Rs\.?\s*([\d,]+)',
        text,
        re.IGNORECASE
    )

    if match:
        return int(
            match.group(1).replace(",", "")
        )

    return None


# ============================================================
# GET STORE NAME FROM URL
# ============================================================

def get_store_name(url):

    if not url:
        return "Unknown"

    try:

        domain = urlparse(url).netloc.lower()

        domain = domain.replace(
            "www.",
            ""
        )

        # Known Indian shopping websites
        stores = {
            "amazon.in": "Amazon",
            "flipkart.com": "Flipkart",
            "croma.com": "Croma",
            "reliancedigital.in": "Reliance Digital",
            "vijaysales.com": "Vijay Sales",
            "tatacliq.com": "Tata CLiQ",
            "myntra.com": "Myntra",
            "ajio.com": "AJIO",
            "meesho.com": "Meesho",
            "snapdeal.com": "Snapdeal",
            "paytmmall.com": "Paytm Mall",
            "jiomart.com": "JioMart",
            "cashify.in": "Cashify"
        }

        if domain in stores:
            return stores[domain]

        # If it is a subdomain
        for website, store_name in stores.items():

            if domain.endswith(
                "." + website
            ):
                return store_name

        # Otherwise use domain name
        name = domain.split(".")[0]

        return name.capitalize()

    except Exception:

        return "Unknown"


# ============================================================
# SEARCH PRODUCTS
# ============================================================

def search_products():

    product = product_entry.get().strip()

    if not product:

        messagebox.showwarning(
            "Input Required",
            "Please enter a product name."
        )

        return

    # Clear previous results
    clear_results()

    status_label.config(
        text="Searching..."
    )

    search_button.config(
        state="disabled"
    )

    low_price_button.config(
        state="disabled"
    )

    threading.Thread(
        target=perform_search,
        args=(product,),
        daemon=True
    ).start()


# ============================================================
# PERFORM SEARCH
# ============================================================

def perform_search(product):

    try:

        results = []

        query = (
            product +
            " buy online price India"
        )

        with DDGS() as ddgs:

            search_results = ddgs.text(
                query,
                max_results=20
            )

            for r in search_results:

                title = r.get(
                    "title",
                    "No title"
                )

                url = r.get(
                    "href",
                    ""
                )

                description = r.get(
                    "body",
                    ""
                )

                # Find price
                price = extract_price(
                    title + " " + description
                )

                # Find store
                store = get_store_name(
                    url
                )

                results.append({
                    "store": store,
                    "product": title,
                    "price": price,
                    "url": url
                })

        root.after(
            0,
            display_results,
            results
        )

    except Exception as e:

        root.after(
            0,
            show_error,
            str(e)
        )


# ============================================================
# DISPLAY RESULTS
# ============================================================

def display_results(results):

    global current_results

    current_results = results

    for result in results:

        price = result["price"]

        if price is None:

            price_text = "Price not found"

        else:

            price_text = (
                f"₹{price:,}"
            )

        add_result_row(
            result["store"],
            result["product"],
            price_text,
            result["url"]
        )

    search_button.config(
        state="normal"
    )

    if results:

        low_price_button.config(
            state="normal"
        )

    status_label.config(
        text=f"{len(results)} result(s) found."
    )


# ============================================================
# ADD RESULT ROW
# ============================================================

def add_result_row(
    store,
    product,
    price,
    url
):

    row = tk.Frame(
        results_container,
        bg="white",
        bd=1,
        relief="solid"
    )

    row.pack(
        fill="x",
        pady=3
    )

    # Store
    store_label = tk.Label(
        row,
        text=store,
        bg="white",
        font=("Arial", 11, "bold"),
        anchor="w",
        width=18
    )

    store_label.pack(
        side="left",
        padx=10,
        pady=10
    )

    # Product
    product_label = tk.Label(
        row,
        text=product,
        bg="white",
        anchor="w",
        width=38
    )

    product_label.pack(
        side="left",
        padx=10
    )

    # Price
    price_label = tk.Label(
        row,
        text=price,
        bg="white",
        fg="#D32F2F",
        font=("Arial", 11, "bold"),
        width=15
    )

    price_label.pack(
        side="left",
        padx=10
    )

    # BUY button
    buy_button = tk.Button(
        row,
        text="BUY",
        bg="#1976D2",
        fg="white",
        font=("Arial", 10, "bold"),
        padx=15,
        command=lambda link=url:
            open_product(link)
    )

    buy_button.pack(
        side="right",
        padx=15,
        pady=8
    )


# ============================================================
# LOWEST PRICE
# ============================================================

def sort_low_price():

    if not current_results:

        return

    # Keep only results with a price
    priced_results = [
        r
        for r in current_results
        if r["price"] is not None
    ]

    # Sort cheapest first
    priced_results.sort(
        key=lambda x: x["price"]
    )

    # Clear GUI
    clear_results()

    # Show sorted results
    for result in priced_results:

        add_result_row(
            result["store"],
            result["product"],
            f"₹{result['price']:,}",
            result["url"]
        )

    if priced_results:

        lowest = priced_results[0]["price"]

        lowest_price_label.config(
            text=f"🟢 LOWEST PRICE: ₹{lowest:,}"
        )

        lowest_price_label.config(
            fg="#2E7D32"
        )

        status_label.config(
            text="Results sorted from LOWEST to HIGHEST price."
        )

    else:

        lowest_price_label.config(
            text="No prices found."
        )


# ============================================================
# OPEN PRODUCT
# ============================================================

def open_product(url):

    if url:

        webbrowser.open(
            url
        )


# ============================================================
# CLEAR RESULTS
# ============================================================

def clear_results():

    for widget in results_container.winfo_children():

        widget.destroy()

    lowest_price_label.config(
        text=""
    )


# ============================================================
# ERROR
# ============================================================

def show_error(error):

    search_button.config(
        state="normal"
    )

    low_price_button.config(
        state="disabled"
    )

    status_label.config(
        text="Search failed."
    )

    messagebox.showerror(
        "Error",
        f"Something went wrong:\n\n{error}"
    )


# ============================================================
# GUI
# ============================================================

root = tk.Tk()

root.title(
    "🛒 Price Comparison"
)

root.geometry(
    "1100x700"
)

root.minsize(
    850,
    550
)


# ============================================================
# TITLE
# ============================================================

title_label = tk.Label(
    root,
    text="🛒 PRICE COMPARISON",
    font=("Arial", 25, "bold"),
    fg="#222222"
)

title_label.pack(
    pady=(20, 15)
)


# ============================================================
# SEARCH AREA
# ============================================================

search_frame = tk.Frame(
    root
)

search_frame.pack(
    fill="x",
    padx=30,
    pady=5
)


product_label = tk.Label(
    search_frame,
    text="Product:",
    font=("Arial", 12, "bold")
)

product_label.pack(
    side="left",
    padx=(0, 10)
)


product_entry = tk.Entry(
    search_frame,
    font=("Arial", 13)
)

product_entry.pack(
    side="left",
    fill="x",
    expand=True,
    ipady=7
)


search_button = tk.Button(
    search_frame,
    text="SEARCH",
    font=("Arial", 11, "bold"),
    bg="#1976D2",
    fg="white",
    padx=25,
    pady=8,
    command=search_products
)

search_button.pack(
    side="left",
    padx=10
)


# ============================================================
# LOWEST PRICE BUTTON
# ============================================================

button_frame = tk.Frame(
    root
)

button_frame.pack(
    fill="x",
    padx=30,
    pady=10
)


low_price_button = tk.Button(
    button_frame,
    text="⬇ LOWEST PRICE",
    font=("Arial", 11, "bold"),
    bg="#2E7D32",
    fg="white",
    padx=25,
    pady=8,
    state="disabled",
    command=sort_low_price
)

low_price_button.pack(
    side="left"
)


# ============================================================
# LOWEST PRICE LABEL
# ============================================================

lowest_price_label = tk.Label(
    button_frame,
    text="",
    font=("Arial", 13, "bold")
)

lowest_price_label.pack(
    side="left",
    padx=30
)


# ============================================================
# TABLE HEADER
# ============================================================

header = tk.Frame(
    root,
    bg="#263238"
)

header.pack(
    fill="x",
    padx=30,
    pady=(10, 0)
)


tk.Label(
    header,
    text="STORE",
    bg="#263238",
    fg="white",
    font=("Arial", 11, "bold"),
    width=18,
    anchor="w"
).pack(
    side="left",
    padx=10,
    pady=10
)


tk.Label(
    header,
    text="PRODUCT",
    bg="#263238",
    fg="white",
    font=("Arial", 11, "bold"),
    width=38,
    anchor="w"
).pack(
    side="left",
    padx=10
)


tk.Label(
    header,
    text="PRICE",
    bg="#263238",
    fg="white",
    font=("Arial", 11, "bold"),
    width=15
).pack(
    side="left",
    padx=10
)


tk.Label(
    header,
    text="BUY",
    bg="#263238",
    fg="white",
    font=("Arial", 11, "bold"),
    width=10
).pack(
    side="right",
    padx=20
)


# ============================================================
# SCROLLABLE RESULTS AREA
# ============================================================

results_outer = tk.Frame(
    root
)

results_outer.pack(
    fill="both",
    expand=True,
    padx=30,
    pady=5
)


canvas = tk.Canvas(
    results_outer,
    bg="#F5F5F5",
    highlightthickness=0
)

scrollbar = ttk.Scrollbar(
    results_outer,
    orient="vertical",
    command=canvas.yview
)

results_container = tk.Frame(
    canvas,
    bg="#F5F5F5"
)


results_container.bind(
    "<Configure>",
    lambda event:
        canvas.configure(
            scrollregion=canvas.bbox("all")
        )
)


canvas_window = canvas.create_window(
    (0, 0),
    window=results_container,
    anchor="nw"
)


def resize_container(event):

    canvas.itemconfig(
        canvas_window,
        width=event.width
    )


canvas.bind(
    "<Configure>",
    resize_container
)


canvas.configure(
    yscrollcommand=scrollbar.set
)


canvas.pack(
    side="left",
    fill="both",
    expand=True
)

scrollbar.pack(
    side="right",
    fill="y"
)


# ============================================================
# STATUS
# ============================================================

status_label = tk.Label(
    root,
    text="Enter a product and click SEARCH.",
    anchor="w",
    fg="gray",
    font=("Arial", 10)
)

status_label.pack(
    fill="x",
    padx=30,
    pady=(5, 15)
)


# ============================================================
# ENTER KEY
# ============================================================

product_entry.bind(
    "<Return>",
    lambda event:
        search_products()
)


product_entry.focus()


# ============================================================
# GLOBAL RESULTS
# ============================================================

current_results = []


# ============================================================
# START APPLICATION
# ============================================================

root.mainloop()

